# Testing on Clips

# Preliminaries

In [ ]:
import os
import pickle
import librosa

import numpy as np
import matplotlib.pyplot as plt

import pydub as pyd
import IPython.display as ipd
import tensorflow as tf

In [19]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [14]:
json_file = open('results/CNN_model.json', 'r')
loaded_model_json = json_file.read()
json_file.close()

loaded_model = tf.keras.models.model_from_json(loaded_model_json)
loaded_model.load_weights("results/best_model.weights.h5")

with open('results/scaler.pickle', 'rb') as f:
    scaler = pickle.load(f)

with open('results/encoder.pickle', 'rb') as f:
    encoder = pickle.load(f)

c:\Users\regdi\Documents\SpeechRecog-1\.conda\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.2.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\regdi\Documents\SpeechRecog-1\.conda\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.2.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


# Speech Emotion Recognition Module

In [15]:
def zcr(data,frame_length,hop_length):
    zcr=librosa.feature.zero_crossing_rate(data,frame_length=frame_length,hop_length=hop_length)
#     print(np.squeeze(zcr))
    return np.squeeze(zcr)

def rmse(data,frame_length=2048,hop_length=512):
    rmse=librosa.feature.rms(y=data,frame_length=frame_length,hop_length=hop_length)
    return np.squeeze(rmse)

def mfcc(data, sr, frame_length=2048, hop_length=512, flatten=True):
    mfcc_result = librosa.feature.mfcc(y=data, sr=sr, n_fft=frame_length, hop_length=hop_length)
    return np.squeeze(mfcc_result.T) if not flatten else np.ravel(mfcc_result.T)

def extract_features(data,sr=22050,frame_length=2048,hop_length=512):
    result=np.array([])

    result=np.hstack((
      result,
      zcr(data,frame_length,hop_length),
      rmse(data,frame_length,hop_length),
      mfcc(data,sr,frame_length,hop_length)
    ))

    return result

def get_predict_feat(path, expected_shape=(1, 2376)):
    d, s_rate = librosa.load(path, duration=2.5, offset=0.6)
    res = extract_features(d)

    # Ensure res is reshaped or padded to match the expected shape
    if res.shape != expected_shape:
        flat_size = np.prod(expected_shape)
        if res.size < flat_size:
            # Pad if the size is smaller than expected
            pad_width = (0, flat_size - res.size)
            res = np.pad(res, pad_width=pad_width, mode='constant')
        else:
            # Resize if the size is larger than expected
            res = np.resize(res, expected_shape)

    i_result = scaler.transform(res.reshape(1, -1))
    final_result = np.expand_dims(i_result, axis=2)

    return final_result

def prediction(path1, predicted_emo = []):
    print(path1)
    res = get_predict_feat(path1)
    predictions = loaded_model.predict(res)

    # Get the label names or define them if available
    label_names = list(encoder.categories_[0])

    # Get the index of the label with the highest confidence score
    predicted_label_index = np.argmax(predictions)

    # List to store confidence scores
    confidence_scores = []

    # Display predicted emotion and confidence for each label
    print(f"\nPredicted Emotion: {label_names[predicted_label_index]}")
    predicted_emo.append(label_names[predicted_label_index])
    for label_index, label_name in enumerate(label_names):
        confidence_score = predictions[0][label_index]
        confidence_score = 0 if confidence_score < 0.001 else confidence_score
        confidence_scores.append({'label': label_name, 'confidence': confidence_score})

    print("\n")

    sorted_confidence_scores = sorted(confidence_scores, key=lambda x: x['confidence'], reverse=True)

    return sorted_confidence_scores


# Test

In [21]:
def generate_confusion_matrix(base_folder, plot_title="Confusion Matrix: Predicted vs Actual Emotions"):
    """
    Generates and plots a confusion matrix from audio files in a flat folder.
    
    Args:
        base_folder (str): Path to folder with audio files named like 'emotion_##.wav'
        plot_title (str): Title for the confusion matrix plot (default provided)
    """
    y_true = []
    y_pred = []

    for filename in os.listdir(base_folder):
        if filename.endswith(".wav") or filename.endswith(".mp3"):
            try:
                # Extract true label from filename
                true_label = filename.split("_")[0].lower()
                file_path = os.path.join(base_folder, filename)

                # Predict using your model
                result = prediction(file_path)
                predicted_label = result[0]['label'].lower()

                y_true.append(true_label)
                y_pred.append(predicted_label)

            except Exception as e:
                print(f"Error processing {filename}: {e}")

    if not y_true or not y_pred:
        print("No predictions were made. Please check your data or prediction function.")
        return

    # Create and display confusion matrix
    labels = sorted(set(y_true + y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)

    plt.figure(figsize=(10, 8))
    disp.plot(cmap='Blues', xticks_rotation=45)
    plt.title(plot_title)
    plt.grid(False)
    plt.show()


In [ ]:
generate_confusion_matrix("Filipino Clips (2.5s)", plot_title="Filipino Dataset (2.5s)")

Filipino Clips (2.5s)\angry_1.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step

Predicted Emotion: disgust


Filipino Clips (2.5s)\angry_10.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step

Predicted Emotion: sad


Filipino Clips (2.5s)\angry_11.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step

Predicted Emotion: sad


Filipino Clips (2.5s)\angry_12.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step

Predicted Emotion: happy


Filipino Clips (2.5s)\angry_13.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step

Predicted Emotion: disgust


Filipino Clips (2.5s)\angry_14.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step

Predicted Emotion: disgust


Filipino Clips (2.5s)\angry_15.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step

Predicted Emotion: disgust


Filipino Clips (2.5s)\angry_16.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step

Predicted Emotion: surprise


Filipino Clips (2.5s)\angry_17.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step

Predicted Emotion: angry


Filipino Clips (2.5s)\angry_18.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step

P